In [ ]:
import sys
import pandas as pd
import numpy as np
import random

random.seed(42)
np.random.seed(42)
rng = np.random.RandomState(42)

df = pd.read_csv('random_forest_dataset.csv')

keep = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Survived']
df = df[keep]

for col in ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'Survived']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

df['Sex'] = df['Sex'].map({'male': 0, 'female': 1}).fillna(0).astype(int)
df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0).astype(int)


df = df.sample(frac=1, random_state=217).reset_index(drop=True)
split = int(0.8 * len(df))
train_df, test_df = df.iloc[:split], df.iloc[split:]
X_train, y_train = train_df.drop('Survived', axis=1).values, train_df['Survived'].values
X_test, y_test = test_df.drop('Survived', axis=1).values, test_df['Survived'].values

def entropy(y):
    if len(y) == 0:
        return 0
    p = np.mean(y)
    if p in [0, 1]:
        return 0
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

def information_gain(y, y_left, y_right):
    n = len(y)
    if n == 0:
        return 0
    return entropy(y) - (len(y_left)/n)*entropy(y_left) - (len(y_right)/n)*entropy(y_right)

def best_split(X, y, max_features):
    n_features = X.shape[1]
    features = rng.choice(n_features, max_features, replace=False)
    best_ig = -1
    best_feature, best_value = None, None
    for f in features:
        vals = np.unique(X[:, f])
        for v in vals:
            left_mask = X[:, f] <= v
            right_mask = ~left_mask
            if left_mask.sum() == 0 or right_mask.sum() == 0:
                continue
            y_left, y_right = y[left_mask], y[right_mask]
            ig = information_gain(y, y_left, y_right)
            if ig > best_ig:
                best_ig, best_feature, best_value = ig, f, v
    return best_feature, best_value

class TreeNode:
    def __init__(self, feature=None, value=None, left=None, right=None, label=None):
        self.feature = feature
        self.value = value
        self.left = left
        self.right = right
        self.label = label

def build_tree(X, y, depth=0, max_depth=10, min_samples_split=2, max_features=3):
    if len(y) < min_samples_split or depth >= max_depth or len(set(y)) == 1:
        return TreeNode(label=int(np.round(np.mean(y))))
    feat, val = best_split(X, y, max_features)
    if feat is None:
        return TreeNode(label=int(np.round(np.mean(y))))
    left_mask = X[:, feat] <= val
    right_mask = ~left_mask
    if left_mask.sum() == 0 or right_mask.sum() == 0:
        return TreeNode(label=int(np.round(np.mean(y))))
    left = build_tree(X[left_mask], y[left_mask], depth + 1, max_depth, min_samples_split, max_features)
    right = build_tree(X[right_mask], y[right_mask], depth + 1, max_depth, min_samples_split, max_features)
    return TreeNode(feature=feat, value=val, left=left, right=right)

def predict_tree(node, x):
    while node.label is None:
        if x[node.feature] <= node.value:
            node = node.left
        else:
            node = node.right
    return node.label

class RandomForest:
    def __init__(self, n_estimators=100, max_depth=10, min_samples_split=2, max_features=3):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []
        self.oob_sets = []

    def bootstrap(self, X, y):
        n = len(X)
        idx = rng.choice(n, n, replace=True)
        oob = np.setdiff1d(np.arange(n), idx)
        return X[idx], y[idx], oob

    def fit(self, X, y):
        self.trees, self.oob_sets = [], []
        for _ in range(self.n_estimators):
            Xb, yb, oob = self.bootstrap(X, y)
            tree = build_tree(Xb, yb, 0, self.max_depth, self.min_samples_split, self.max_features)
            self.trees.append(tree)
            self.oob_sets.append(oob)

    def predict_one(self, x):
        preds = [predict_tree(t, x) for t in self.trees]
        return int(round(np.mean(preds)))

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

    def oob_score(self, X, y):
        correct = 0
        total = 0
        for i in range(len(X)):
            preds = [predict_tree(self.trees[j], X[i]) for j in range(self.n_estimators) if i in self.oob_sets[j]]
            if preds:
                maj = int(round(np.mean(preds)))
                total += 1
                if maj == y[i]:
                    correct += 1
        return correct / total if total > 0 else 0


rf = RandomForest(n_estimators=100, max_depth=10, min_samples_split=2, max_features=3)
rf.fit(X_train, y_train)

oob_est = rf.oob_score(X_train, y_train)
test_acc = np.mean(rf.predict(X_test) == y_test)


print(f"OOB estimate: {oob_est:.2f}")
print(f"Testing accuracy: {test_acc:.2f}")